# 01. Data Cleaning & Experiment Design

**Project:** Discount Strategy A/B Test Evaluation
**Goal:** Prepare a clean customer-level dataset, assign groups (control/treatment), and simulate the discount effect.

**Inputs:** `data/raw/online_retail_II.xlsx`
**Outputs:** `data/processed/transactions_clean.parquet`, `data/processed/customers.parquet`, `data/processed/experiment_{weak|medium|strong}.parquet`

In [2]:
import hashlib
import pathlib
import warnings

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

ROOT = pathlib.Path("..").resolve()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
SALT = "discount_2024Q4"

## 1. Load raw data

The file contains two sheets: `Year 2009-2010` and `Year 2010-2011`. We are merging them.

In [3]:
xlsx_path = RAW / "online_retail_II.xlsx"
assert xlsx_path.exists(), f"Put the file in {xlsx_path}"

sheets = ["Year 2009-2010", "Year 2010-2011"]
raw = pd.concat(
    [pd.read_excel(xlsx_path, sheet_name=s) for s in sheets],
    ignore_index=True,
)

print("shape:", raw.shape)
display(raw.head())
print(raw.dtypes)

shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.950,"13,085.000",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750,"13,085.000",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750,"13,085.000",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.100,"13,085.000",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.250,"13,085.000",United Kingdom


Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object


## 2. Cleaning

Rules:
- remove cancellations (`Invoice` starts with `C`);
- `Quantity > 0`, `Price > 0`;
- remove guest customers (`Customer ID is null`);
- remove service-related StockCodes;
- exclude the top 1% of largest orders (wholesalers).

In [4]:
SERVICE_CODES = {
    "POST","D","M","BANK CHARGES","AMAZONFEE","CRUK","PADS","DOT","S",
    "C2","ADJUST","ADJUST2","TEST001","TEST002","gift_0001_20",
}

def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().replace(" ", "_").lower() for c in df.columns]
    df["invoice"] = df["invoice"].astype(str)

    n0 = len(df)
    df = df[~df["invoice"].str.startswith("C")];        print(f"− cancellations   : {n0-len(df):>8,}")
    n1 = len(df)
    df = df[df["quantity"] > 0];                        print(f"− qty<=0          : {n1-len(df):>8,}")
    n2 = len(df)
    df = df[df["price"] > 0];                           print(f"− price<=0        : {n2-len(df):>8,}")
    n3 = len(df)
    df = df[df["customer_id"].notna()];                 print(f"− guests (NaN ID) : {n3-len(df):>8,}")
    n4 = len(df)
    df = df[~df["stockcode"].isin(SERVICE_CODES)];      print(f"− service codes   : {n4-len(df):>8,}")

    df["customer_id"] = df["customer_id"].astype(int)
    df["invoicedate"] = pd.to_datetime(df["invoicedate"])
    df["revenue"] = df["quantity"] * df["price"]

    # filtering out wholesalers
    order_sum = df.groupby("invoice")["revenue"].sum()
    thr = order_sum.quantile(0.99)
    big = order_sum[order_sum > thr].index
    n5 = len(df)
    df = df[~df["invoice"].isin(big)]
    print(f"− top-1% invoices : {n5-len(df):>8,}")
    print(f"Total removed     : {n0-len(df):>8,} ({100*(n0-len(df))/n0:.1f}%)")

    return df.reset_index(drop=True)

tx = clean(raw)
print("\nshape:", tx.shape)

− cancellations   :   19,494
− qty<=0          :    3,457
− price<=0        :    2,750
− guests (NaN ID) :  236,121
− service codes   :    2,915
− top-1% invoices :   17,734
Total removed     :  282,471 (26.5%)

shape: (784900, 9)


In [5]:
assert (tx["revenue"] > 0).all()
assert tx["customer_id"].notna().all()
assert not tx["invoice"].str.startswith("C").any()

print("Period          :", tx["invoicedate"].min().date(), "→", tx["invoicedate"].max().date())
print("Lines           :", f"{len(tx):,}")
print("Unique orders:", f"{tx['invoice'].nunique():,}")
print("Unique customers:", f"{tx['customer_id'].nunique():,}")
print("\nTop 5 countries:")
print(tx["country"].value_counts().head())

Period          : 2009-12-01 → 2011-12-09
Lines           : 784,900
Unique orders: 36,228
Unique customers: 5,841

Top 5 countries:
country
United Kingdom    713687
Germany            15809
EIRE               14366
France             12826
Spain               3578
Name: count, dtype: int64


In [16]:
tx["stockcode"]   = tx["stockcode"].astype(str)
tx["description"] = tx["description"].astype(str)
tx["invoice"]     = tx["invoice"].astype(str)
tx.to_parquet(PROC / "transactions_clean.parquet", index=False)
print("saved:", PROC / "transactions_clean.parquet")

saved: C:\Users\Yana\Desktop\discount-ab-test-evaluation\data\processed\transactions_clean.parquet


## 3. Customer-level features (RFM-like)

One row = one customer. This is the unit of randomization for the future experiment.

In [7]:
snapshot = tx["invoicedate"].max() + pd.Timedelta(days=1)
print("snapshot:", snapshot)

g = tx.groupby("customer_id")
customers = pd.DataFrame({
    "first_purchase":  g["invoicedate"].min(),
    "last_purchase":   g["invoicedate"].max(),
    "frequency":       g["invoice"].nunique(),
    "monetary":        g["revenue"].sum(),
    "total_items":     g["quantity"].sum(),
    "n_products":      g["stockcode"].nunique(),
}).reset_index()

customers["aov"]            = customers["monetary"] / customers["frequency"]
customers["recency"]        = (snapshot - customers["last_purchase"]).dt.days
customers["tenure_days"]    = (snapshot - customers["first_purchase"]).dt.days

# modal country
country_mode = g["country"].agg(lambda s: s.mode().iat[0])
customers = customers.merge(country_mode.rename("country"), on="customer_id")
customers["is_uk"] = customers["country"].eq("United Kingdom")

# segments
customers["is_new"] = customers["tenure_days"] <= 90
med = customers["monetary"].median()
customers["value_segment"] = np.where(customers["monetary"] > med, "high", "low")

print("Customers:", len(customers))
display(customers.describe())

snapshot: 2011-12-10 12:50:00
Customers: 5841


,customer_id,first_purchase,last_purchase,frequency,monetary,total_items,n_products,aov,recency,tenure_days
count,"5,841.000",5841,5841,"5,841.000","5,841.000","5,841.000","5,841.000","5,841.000","5,841.000","5,841.000"
mean,"15,321.882",2010-08-22 19:54:04.262968,2011-05-23 14:59:47.755521,6.202,"2,476.499","1,481.752",81.463,346.213,200.388,474.173
min,"12,346.000",2009-12-01 07:45:00,2009-12-01 10:49:00,1.000,2.900,1.000,1.000,2.900,1.000,1.000
25%,"13,841.000",2010-02-11 09:41:00,2010-11-25 14:56:00,1.000,344.000,191.000,19.000,179.718,26.000,312.000
50%,"15,323.000",2010-06-28 12:58:00,2011-09-05 14:13:00,3.000,874.940,491.000,45.000,283.110,95.000,529.000
75%,"16,804.000",2011-02-01 11:31:00,2011-11-14 12:49:00,7.000,"2,250.110","1,364.000",103.000,416.350,379.000,667.000
max,"18,287.000",2011-12-09 12:16:00,2011-12-09 12:50:00,368.000,"255,765.310","149,040.000","2,536.000","3,528.340",739.000,739.000
std,"1,714.048",NaN,NaN,12.427,"7,111.752","4,549.355",113.926,276.750,208.519,223.129


In [8]:
print("UK share       :", f"{customers['is_uk'].mean():.1%}")
print("New (<90) share:", f"{customers['is_new'].mean():.1%}")
print("High-value    :", f"{(customers['value_segment']=='high').mean():.1%}")
print("\nMonetary quantiles:")
print(customers["monetary"].quantile([0.1,0.25,0.5,0.75,0.9,0.99]))

UK share       : 91.2%
New (<90) share: 10.2%
High-value    : 50.0%

Monetary quantiles:
0.100      162.500
0.250      344.000
0.500      874.940
0.750    2,250.110
0.900    5,362.980
0.990   24,991.936
Name: monetary, dtype: float64


In [9]:
customers.to_parquet(PROC / "customers.parquet", index=False)
print("saved:", PROC / "customers.parquet")

saved: C:\Users\Yana\Desktop\discount-ab-test-evaluation\data\processed\customers.parquet


## 4. Experiment design

**Randomization unit:** `customer_id`
**Split:** 50/50, deterministic hash `md5(SALT + customer_id)`
**Horizon:** entire historical period (we simulate future behavior based on the customer table)

**Metrics:**
- Primary: Revenue per customer (RPC)
- Secondary: Conversion Rate, AOV
- Guardrail: return rate

**Effect scenarios:**

| Scenario | Δ Conversion | Δ AOV | Δ Return rate |
|---|---|---|---|
| weak   | +5%  | −2% | +1.0 pp |
| medium | +12% | 0% | +1.0 pp |
| strong | +20% | +3% | +1.0 pp |

In [10]:
def assign_group(customer_id: int) -> str:
    h = hashlib.md5(f"{SALT}:{customer_id}".encode()).hexdigest()
    return "treatment" if int(h, 16) % 100 < 50 else "control"

customers["group"] = customers["customer_id"].apply(assign_group)
print(customers["group"].value_counts())
print(customers["group"].value_counts(normalize=True).round(4))

group
treatment    2937
control      2904
Name: count, dtype: int64
group
treatment   0.503
control     0.497
Name: proportion, dtype: float64


In [11]:
def simulate_outcomes(customers: pd.DataFrame,
                      scenario: str = "medium",
                      seed: int = RANDOM_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    df = customers.copy()
    is_t = (df["group"] == "treatment").values

    # baseline conversion probabilities within the window
    base_p = 1 / (1 + np.exp(-(-1.5 + 0.02*df["frequency"] - 0.005*df["recency"])))

    params = {
        "weak":   dict(conv_lift=0.05, aov_lift=-0.02, return_delta=0.010),
        "medium": dict(conv_lift=0.12, aov_lift= 0.00, return_delta=0.010),
        "strong": dict(conv_lift=0.20, aov_lift= 0.03, return_delta=0.010),
    }[scenario]

    # conversion
    p = base_p.copy()
    p[is_t] = np.clip(p[is_t] * (1 + params["conv_lift"]), 0, 1)
    df["converted"] = (rng.random(len(df)) < p).astype(int)

    # AOV via log-normal
    sigma = 0.6
    aov = df["aov"].clip(lower=5).values * np.exp(rng.normal(0, sigma, len(df)))
    aov[is_t] *= (1 + params["aov_lift"])
    df["aov_sim"] = np.where(df["converted"] == 1, aov, 0.0)

    # revenue per customer
    df["revenue_sim"] = df["aov_sim"] * df["converted"]

    # guardrail
    base_return = 0.04 + 0.001 * (df["frequency"] < 3).values
    ret = base_return + rng.normal(0, 0.01, len(df))
    ret[is_t] += params["return_delta"]
    df["return_rate"] = np.clip(ret, 0, 0.25)

    df["scenario"] = scenario
    return df

In [12]:
scenarios = {}
for sc in ["weak", "medium", "strong"]:
    exp = simulate_outcomes(customers, scenario=sc)
    exp.to_parquet(PROC / f"experiment_{sc}.parquet", index=False)
    scenarios[sc] = exp
    print(f"{sc:>6s} saved | group counts: {exp['group'].value_counts().to_dict()}")

  weak saved | group counts: {'treatment': 2937, 'control': 2904}
medium saved | group counts: {'treatment': 2937, 'control': 2904}
strong saved | group counts: {'treatment': 2937, 'control': 2904}


## 5. A/A test (sanity check)

We verify that the groups are balanced across **pre-treatment** features.
Outcome metrics (`converted`, `revenue_sim`) must **not** be checked here — the effect is already present in them.

In [13]:
exp = scenarios["medium"]
PRE_COLS = ["frequency","monetary","aov","recency","tenure_days","total_items","n_products","is_uk","is_new"]

rows = []
for col in PRE_COLS:
    a = exp.loc[exp.group=="control", col]
    b = exp.loc[exp.group=="treatment", col]
    if exp[col].dtype == bool:
        a, b = a.astype(int), b.astype(int)
    t, p = stats.ttest_ind(a, b, equal_var=False)
    rows.append({
        "feature": col,
        "control_mean": a.mean(),
        "treatment_mean": b.mean(),
        "diff_pct": 100*(b.mean()-a.mean())/a.mean() if a.mean() else np.nan,
        "p_value": p,
    })

aa = pd.DataFrame(rows)
aa["flag"] = np.where(aa["p_value"] < 0.05, "⚠️", "ok")
display(aa)

,feature,control_mean,treatment_mean,diff_pct,p_value,flag
0,frequency,6.166,6.238,1.162,0.826,ok
1,monetary,"2,472.537","2,480.417",0.319,0.966,ok
2,aov,350.770,341.707,-2.584,0.211,ok
3,recency,198.037,202.712,2.360,0.392,ok
4,tenure_days,472.614,475.714,0.656,0.596,ok
5,total_items,"1,492.556","1,471.069",-1.440,0.857,ok
6,n_products,80.490,82.424,2.403,0.516,ok
7,is_uk,0.911,0.913,0.222,0.785,ok
8,is_new,0.104,0.099,-5.039,0.506,ok


In [15]:
n_flagged = (aa["p_value"] < 0.05).sum()
print(f"Features with p<0.05: {n_flagged} out of {len(aa)}")
if n_flagged == 0:
    print("✅ Assignment is balanced — the effect can be analyzed.")
else:
    print("⚠️ There is an imbalance — consider checking the hash or changing the SALT.")

Features with p<0.05: 0 out of 9
✅ Assignment is balanced — the effect can be analyzed.
